# Targeting: Does Personalization Work?
## P@S Class 20: Replication notebook

**Data labels:** Sections 1-3 load **REAL** replication data from published packages. Section 4 uses **SIMULATED** data matching Aggarwal et al.'s reported estimates (individual-level voter file data is not public).

| Section | Paper | Data source | N |
|---------|-------|-------------|---|
| 1 | Coppock, Hill & Vavreck (2020) | Harvard Dataverse doi:10.7910/DVN/TN7KWR | 34,000 respondents |
| 2 | Tappin et al. (2023) | OSF osf.io/t3dhe | 16,887 participants |
| 3 | Simchon et al. (2024) | OSF osf.io/5w3ct | 1,243 participants |
| 4 | Aggarwal et al. (2023) | **SIMULATED** from reported estimates | 2,000,000 voters |

**The question:** The ATE of political persuasion is ~0 (Class 19). Can you target subgroups to rescue the null?

In [ ]:
!pip install -q numpy matplotlib scipy pandas scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.lines import Line2D

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
BLUE, RED, GREEN, GRAY = '#2980b9', '#c0392b', '#27ae60', '#95a5a6'
np.random.seed(42)

DATA = 'https://raw.githubusercontent.com/Persuasion-at-Scale/targeting/main/data/'
print("Setup complete.")

---
## Section 1: Coppock, Hill & Vavreck (2020) [REAL]

**Data:** Pre-computed treatment effect estimates from the authors' replication package. 354 conditional average treatment effects (CATEs) for favorability, stratified by respondent party, ad type, and survey wave. Plus 59 sample-average treatment effects (SATEs) across weekly survey waves.

**Replication targets:**
- **Figure 1:** SATEs over time + random-effects meta-analytic pooled estimate
- **Figure 2:** CATEs by respondent party x ad partisanship, pooled via random-effects meta-analysis
- **Table 1 logic:** Meta-regression testing whether ANY moderator (party, ad type, battleground, timing) predicts effect size

In [ ]:
# [REAL] Load pre-computed CATEs and SATEs from Coppock replication package
cates = pd.read_csv(DATA + 'coppock-favorability-cates.csv')
sates = pd.read_csv(DATA + 'coppock-favorability-sates.csv')
print(f"[REAL] Loaded {len(cates)} CATEs (favorability) and {len(sates)} SATEs")
print(f"CATE columns: pid_3_pre, ad_match, pro_democrat, estimate, std.error, ...")
print(f"Parties: {sorted(cates['pid_3_pre'].unique())}")
print(f"Ad types: {sorted(cates['pro_democrat'].unique())}")
print(f"SATE date range: {sates['date'].min()} to {sates['date'].max()}")

In [ ]:
# Replication of Figure 1: SATEs over time + meta-analytic pooled estimate
# Random-effects meta-analysis (DerSimonian-Laird)

def dl_meta(estimates, ses):
    """DerSimonian-Laird random-effects meta-analysis."""
    w = 1.0 / ses**2
    theta_fe = np.average(estimates, weights=w)
    Q = np.sum(w * (estimates - theta_fe)**2)
    k = len(estimates)
    c = w.sum() - (w**2).sum() / w.sum()
    tau2 = max((Q - (k - 1)) / c, 0)
    w_re = 1.0 / (ses**2 + tau2)
    theta_re = np.average(estimates, weights=w_re)
    se_re = np.sqrt(1.0 / w_re.sum())
    return theta_re, se_re, tau2, Q, k

# Pool all SATEs
theta, se, tau2, Q, k = dl_meta(sates['estimate'].values, sates['std.error'].values)
p_val = 2 * (1 - stats.norm.cdf(abs(theta / se)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [3, 1]})

# Panel A: SATEs over time
ax = axes[0]
sates_sorted = sates.sort_values('date')
dates = pd.to_datetime(sates_sorted['date'])
ax.errorbar(dates, sates_sorted['estimate'], yerr=1.96*sates_sorted['std.error'],
            fmt='o', color=BLUE, markersize=4, capsize=2, alpha=0.7, linewidth=1)
ax.axhline(0, color='black', linestyle='--', alpha=0.3)
ax.set_xlabel('Survey Date (2016)')
ax.set_ylabel('Treatment Effect (favorability, 1-5)')
ax.set_title(f'[REAL] Coppock Fig. 1: Weekly SATEs (k={k})')
ax.tick_params(axis='x', rotation=45)

# Panel B: Meta-analytic estimate
ax2 = axes[1]
ax2.errorbar([0], [theta], yerr=[[1.96*se], [1.96*se]],
             fmt='D', color='black', markersize=12, capsize=8, linewidth=2)
ax2.axhline(0, color='black', linestyle='--', alpha=0.3)
ax2.set_xlim(-0.5, 0.5)
ax2.set_xticks([0])
ax2.set_xticklabels(['Pooled\n(DL)'])
ax2.set_title(f'Meta-analytic\nestimate')
ax2.text(0.3, theta, f'{theta:.3f}\n({se:.3f})', fontsize=11, va='center')

plt.suptitle(f'Pooled effect = {theta:.3f} (SE = {se:.3f}, p = {p_val:.3f})', fontsize=13)
plt.tight_layout()
plt.show()

print(f"DerSimonian-Laird random-effects meta-analysis:")
print(f"  Pooled effect: {theta:.4f} (SE = {se:.4f})")
print(f"  95% CI: [{theta - 1.96*se:.4f}, {theta + 1.96*se:.4f}]")
print(f"  p-value: {p_val:.4f}")
print(f"  Q({k-1}) = {Q:.2f}, tau^2 = {tau2:.4f}")

**What you should see [REAL data]:** Weekly treatment effect estimates scattered around zero with wide CIs. The meta-analytic diamond sits near zero with a tight CI. This replicates Coppock et al.'s Figure 1: the pooled effect of political advertising on favorability is essentially zero.

In [ ]:
# Replication of Figure 2: CATEs by respondent party x ad partisanship
# Pool CATEs within each (party, ad_direction) cell using DL meta-analysis

fig, ax = plt.subplots(figsize=(10, 6))

parties = ['Democrat', 'Independent', 'Republican']
ad_labels = {1: 'Pro-Democratic ad', 0: 'Pro-Republican ad'}
colors = {1: BLUE, 0: RED}
offsets = {1: 0.15, 0: -0.15}

y_pos = 0
y_ticks, y_labels = [], []

for party in parties:
    for pro_dem in [1, 0]:
        sub = cates[(cates['pid_3_pre'] == party) & (cates['pro_democrat'] == pro_dem)]
        if len(sub) < 2:
            continue
        theta, se, tau2, Q, k = dl_meta(sub['estimate'].values, sub['std.error'].values)

        color = colors[pro_dem]
        ax.plot([theta - 1.96*se, theta + 1.96*se], [y_pos, y_pos],
                color=color, linewidth=2, alpha=0.8)
        ax.plot(theta, y_pos, 'D', color=color, markersize=8, zorder=5)
        ax.text(theta + 1.96*se + 0.02, y_pos,
                f'{theta:.3f} ({se:.3f})', fontsize=9, va='center', color=color)

        y_ticks.append(y_pos)
        y_labels.append(f'{party}: {ad_labels[pro_dem]}')
        y_pos += 1
    y_pos += 0.5

ax.axvline(0, color='black', linestyle='--', alpha=0.3)
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=10)
ax.set_xlabel('Pooled CATE on Favorability (random-effects)')
ax.set_title('[REAL] Coppock Fig. 2: CATEs by Respondent Party x Ad Direction\n'
             'Random-effects meta-analysis within each cell')
ax.legend(handles=[
    Line2D([0], [0], marker='D', color='w', markerfacecolor=BLUE, markersize=10, label='Pro-Democratic ad'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor=RED, markersize=10, label='Pro-Republican ad')
], loc='lower right')
plt.tight_layout()
plt.show()

**What you should see [REAL data]:** Six pooled CATEs (3 parties x 2 ad directions). Most CIs include or hover near zero. There is no large, consistent subgroup effect. Democrats are not dramatically moved by pro-Democratic ads; Republicans are not dramatically moved by pro-Republican ads. This replicates the paper's Figure 2: the null ATE is not hiding large offsetting heterogeneous effects.

In [ ]:
# Replication of Table 1 logic: meta-regression
# Does ANY moderator predict effect size?
# Moderators: party, ad valence (attack/positive), battleground, timing, PAC sponsor
import statsmodels.api as sm

# Use CATEs as "studies" with inverse-variance weights
reg_data = cates.dropna(subset=['estimate', 'std.error', 'attack_d', 'democrat_d',
                                 'independent_d', 'battleground_d', 'pac_d', 'date_d']).copy()
reg_data['weight'] = 1.0 / reg_data['std.error']**2

X = reg_data[['attack_d', 'democrat_d', 'independent_d', 'battleground_d', 'pac_d', 'date_d']]
X = sm.add_constant(X)
y = reg_data['estimate']
w = reg_data['weight']

wls = sm.WLS(y, X, weights=w).fit()

print("=== Meta-Regression: What Moderates Ad Effects? ===")
print(f"N = {len(reg_data)} CATEs, weighted by inverse variance\n")
print(f"{'Moderator':20s} {'Coef':>8s} {'SE':>8s} {'p':>8s}")
print("-" * 48)
for var in wls.params.index:
    label = var.replace('_d', '').replace('_', ' ').title()
    print(f"{label:20s} {wls.params[var]:8.4f} {wls.bse[var]:8.4f} {wls.pvalues[var]:8.3f}")

print(f"\nR-squared: {wls.rsquared:.4f}")
print(f"F-statistic: {wls.fvalue:.2f}, p = {wls.f_pvalue:.3f}")
print("\nNo moderator is significant: small effects are uniformly small.")

**What you should see [REAL data]:** A meta-regression with ~350 CATEs as observations, weighted by inverse variance. None of the moderators (attack vs. positive, Democrat vs. Republican viewer, battleground state, PAC-sponsored, timing) significantly predict effect size. R-squared is near zero. This replicates the logic of Coppock et al.'s Table 1: there is no systematic pattern in what makes ads more or less effective.

---
## Section 2: Tappin et al. (2023) [REAL]

**Data:** 16,887 participants from the experiment phase (OSF). Three policy issues (immigration, UBI, local). Participants assigned to control, naive messaging, single-best message, or targeting (personalized by covariates). Plus pre-computed precision-weighted pooled estimates from the authors' analysis.

**Replication targets:**
- Condition means (targeting vs. single-best vs. naive vs. control) by issue
- Precision-weighted pooled targeting returns (the paper's headline numbers)
- R-squared analysis: party only vs. party + demographics + ideology

In [ ]:
# [REAL] Load Tappin experiment data and pre-computed pooled estimates
tp = pd.read_csv(DATA + 'tappin-experiment.csv')
pooled = pd.read_csv(DATA + 'tappin-pooled-means.csv')

print(f"[REAL] Loaded {len(tp):,} participants across 3 experiments")
print(f"[REAL] Loaded {len(pooled)} precision-weighted pooled comparisons\n")
print("Pooled targeting returns (from authors' meta-analysis):")
print(pooled.to_string(index=False))

In [ ]:
# Replication: condition means by experiment
# Compute treatment-group means with 95% CIs, matching the paper's analysis

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
issues = [('imm_treat', 'Immigration\n(Study 1)'), ('ubi_treat', 'UBI\n(Study 1)'), ('peoria_treat', 'Local Policy\n(Study 2)')]
scale_factor = {('imm_treat', 6): 6, ('ubi_treat', 6): 6, ('peoria_treat', 4): 4}

for ax, (treat_col, title) in zip(axes, issues):
    conditions = ['control', 'naive', 'single_best', 'targeting']
    cond_labels = ['Control', 'Naive', 'Single\nBest', 'Targeted']
    means, cis, ns = [], [], []

    for cond in conditions:
        subset = tp[tp[treat_col] == cond]['persuade'].dropna()
        if len(subset) == 0:
            means.append(np.nan); cis.append(0); ns.append(0)
        else:
            means.append(subset.mean())
            cis.append(1.96 * subset.sem())
            ns.append(len(subset))

    x = range(len(conditions))
    bars = ax.bar(x, means, yerr=cis, capsize=5,
                  color=[GRAY, BLUE, GREEN, RED], alpha=0.8, edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(cond_labels, fontsize=10)
    ax.set_title(f'[REAL] {title}')
    ax.set_ylabel('Persuasion (1-7)' if ax == axes[0] else '')
    for i, n in enumerate(ns):
        if n > 0:
            ax.text(i, means[i] - 0.15, f'n={n:,}', ha='center', fontsize=8, color='white', fontweight='bold')

plt.suptitle('Tappin et al. (2023): Condition Means with 95% CIs', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Replication: precision-weighted pooled targeting returns (the paper's headline)
# These are the authors' own meta-analytic estimates pooling across experiments

fig, ax = plt.subplots(figsize=(10, 4))

comparisons = pooled['comparison'].values
estimates = pooled['b'].values
lwr = pooled['lwr'].values
upr = pooled['upr'].values

# Color: green if CI excludes zero (significant), gray otherwise
colors_bar = [GREEN if l > 0 or u < 0 else GRAY for l, u in zip(lwr, upr)]

for i, (comp, est, lo, hi, col) in enumerate(zip(comparisons, estimates, lwr, upr, colors_bar)):
    ax.plot([lo, hi], [i, i], color=col, linewidth=2.5, alpha=0.8)
    ax.plot(est, i, 'D', color=col, markersize=10, zorder=5)
    sig = '*' if lo > 0 or hi < 0 else ''
    ax.text(max(hi, 0.05) + 0.02, i, f'{est:.3f} [{lo:.3f}, {hi:.3f}]{sig}',
            fontsize=9, va='center')

ax.axvline(0, color='black', linestyle='--', alpha=0.4)
ax.set_yticks(range(len(comparisons)))
ax.set_yticklabels([c.replace('vs.', 'vs.\n') for c in comparisons], fontsize=10)
ax.set_xlabel('Precision-Weighted Pooled Effect (persuasion scale units)')
ax.set_title('[REAL] Tappin et al. (2023): Pooled Targeting Returns\n'
             'From authors\' inverse-variance meta-analysis across experiments')
plt.tight_layout()
plt.show()

print("Key comparison: targeting vs. single_best")
row = pooled[pooled['comparison'] == 'target vs. single_best'].iloc[0]
print(f"  Pooled effect: {row['b']:.3f} [{row['lwr']:.3f}, {row['upr']:.3f}]")
sig = "SIGNIFICANT" if row['lwr'] > 0 else "NOT significant"
print(f"  {sig} at 95% level")
print(f"\nTargeting beats naive messaging, but the gain over the SINGLE BEST message is small.")

In [ ]:
# How much does party explain vs. everything else?
from sklearn.linear_model import LinearRegression

imm = tp[tp['imm_treat'].isin(['targeting', 'single_best', 'naive'])].copy()
imm = imm.dropna(subset=['persuade', 'party'])

# party: 1=Democrat, 2=Republican, 3/4=Independent/Other
imm['is_dem'] = (imm['party'] == 1).astype(float)
imm['is_rep'] = (imm['party'] == 2).astype(float)

# Nested models
specs = [
    ('Party only', ['is_dem', 'is_rep']),
    ('+ Gender, Age', ['is_dem', 'is_rep', 'gender_sr', 'age_sr']),
    ('+ Ideology', ['is_dem', 'is_rep', 'gender_sr', 'age_sr', 'ideo']),
]

r2s, labels = [], []
for label, cols in specs:
    sub = imm.dropna(subset=cols + ['persuade'])
    X = sub[cols].values
    y = sub['persuade'].values
    m = LinearRegression().fit(X, y)
    r2s.append(m.score(X, y))
    labels.append(label)

fig, ax = plt.subplots(figsize=(8, 5))
colors_r2 = [RED, BLUE, GREEN]
bars = ax.bar(labels, r2s, color=colors_r2, alpha=0.8, edgecolor='white', width=0.5)
for bar, r2 in zip(bars, r2s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'R\u00b2 = {r2:.4f}', ha='center', fontsize=13, fontweight='bold')

ax.set_ylabel('R\u00b2 (variance explained)')
ax.set_title('[REAL] Predicting Persuadability: Diminishing Returns\n'
             'Each bar adds more covariates; gains are tiny')
ax.set_ylim(0, max(r2s) * 2.5 if max(r2s) > 0 else 0.02)
plt.tight_layout()
plt.show()

for label, r2 in zip(labels, r2s):
    print(f"{label:20s}: R\u00b2 = {r2:.4f}")
print(f"\nMarginal gain from demographics: {r2s[1] - r2s[0]:.4f}")
print(f"Marginal gain from ideology: {r2s[2] - r2s[1]:.4f}")

**What you should see [REAL data]:**

1. **Condition means:** Targeting performs about the same as single-best across all three issues. The bars are nearly identical.
2. **Pooled targeting returns:** The precision-weighted forest plot shows targeting vs. single-best is a small positive effect, but targeting vs. naive is larger. The practical gain from personalization over a single good message is minimal.
3. **R-squared:** Party explains most of the (tiny) predictable variation. Adding gender, age, and ideology provides negligible marginal improvement.

This is Tappin et al.'s core finding: **party ID is the targeting ceiling.** The exotic psychographic data that Cambridge Analytica claimed was the secret weapon buys almost nothing beyond what every campaign already has from the voter file.

---
## Section 3: Simchon et al. (2024) [REAL]

**Data:** Studies 1a (generic condition, n=440) and 1b (targeted condition, n=803) from OSF. Each participant rated GPT-4-generated persuasive messages on 10 political issues, 6 Likert items per issue (1=Strongly disagree to 5=Strongly agree).

**The question:** If content generation is free, does targeting finally break through?

**What we do:** Convert Likert to numeric, compute per-issue and overall means, test generic vs. targeted with proper standard errors.

In [ ]:
# [REAL] Load and score Simchon et al. (2024) data
s1a = pd.read_csv(DATA + 'simchon-study-1a.csv')
s1b = pd.read_csv(DATA + 'simchon-study-1b.csv')

likert_map = {'Strongly disagree': 1, 'Somewhat disagree': 2,
              'Neither agree nor disagree': 3, 'Somewhat agree': 4, 'Strongly agree': 5}

issues = sorted(set(int(c.split('_')[1]) for c in s1a.columns if c.startswith('ad_')))

def score_study(df, issues, likert_map):
    """Compute per-participant mean agreement across all items."""
    ad_cols = [c for c in df.columns if c.startswith('ad_')]
    scored = df[ad_cols].map(lambda x: likert_map.get(x, np.nan))
    df = df.copy()
    df['mean_agreement'] = scored.mean(axis=1)
    # Per-issue means
    for iss in issues:
        cols = [f'ad_{iss}_{j}' for j in range(1, 7)]
        df[f'issue_{iss}_mean'] = df[cols].map(lambda x: likert_map.get(x, np.nan)).mean(axis=1)
    return df

s1a = score_study(s1a, issues, likert_map)
s1b = score_study(s1b, issues, likert_map)

print(f"[REAL] Study 1a (Generic): {len(s1a)} participants, mean agreement = {s1a['mean_agreement'].mean():.3f}")
print(f"[REAL] Study 1b (Targeted): {len(s1b)} participants, mean agreement = {s1b['mean_agreement'].mean():.3f}")

In [ ]:
# Per-issue comparison + overall test
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Per-issue paired comparison
ax = axes[0]
gen_means, tgt_means = [], []
for i, iss in enumerate(issues):
    g = s1a[f'issue_{iss}_mean'].mean()
    t = s1b[f'issue_{iss}_mean'].mean()
    gen_means.append(g)
    tgt_means.append(t)
    ax.plot([g, t], [i, i], color=GRAY, linewidth=1, alpha=0.5)
    ax.plot(g, i, 'o', color=GREEN, markersize=9, zorder=5)
    ax.plot(t, i, 's', color=RED, markersize=9, zorder=5)

ax.set_yticks(range(len(issues)))
ax.set_yticklabels([f'Issue {iss}' for iss in issues], fontsize=10)
ax.set_xlabel('Mean Agreement (1-5 Likert)')
ax.set_title('[REAL] Per-Issue Comparison')
ax.legend(handles=[
    Line2D([0], [0], marker='o', color='w', markerfacecolor=GREEN, markersize=10, label=f'Generic (n={len(s1a)})'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=RED, markersize=10, label=f'Targeted (n={len(s1b)})')
], loc='lower right')

# Panel 2: participant-level overall means with proper SEs
ax2 = axes[1]
g_vals = s1a['mean_agreement'].dropna()
t_vals = s1b['mean_agreement'].dropna()
t_stat, p_val = stats.ttest_ind(g_vals, t_vals)

means = [g_vals.mean(), t_vals.mean()]
ses = [g_vals.sem(), t_vals.sem()]
bars = ax2.bar(['Generic\n(Study 1a)', 'Targeted\n(Study 1b)'], means,
               yerr=[1.96*s for s in ses], capsize=8,
               color=[GREEN, RED], alpha=0.85, edgecolor='white', width=0.4)
for bar, m, se in zip(bars, means, ses):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{m:.3f}\n(SE={se:.3f})', ha='center', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean Agreement (1-5)')
ax2.set_title(f'[REAL] Participant-Level Means\nt={t_stat:.2f}, p={p_val:.3f}')

plt.suptitle('Simchon et al. (2024): GPT-4 Generic vs. Targeted', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Count how many issues generic wins
gen_wins = sum(1 for g, t in zip(gen_means, tgt_means) if g > t)
print(f"\nGeneric wins on {gen_wins}/{len(issues)} issues")
print(f"Participant-level t-test: t={t_stat:.2f}, p={p_val:.3f}")
print(f"Generic mean: {g_vals.mean():.3f} (SE={g_vals.sem():.4f}, n={len(g_vals)})")
print(f"Targeted mean: {t_vals.mean():.3f} (SE={t_vals.sem():.4f}, n={len(t_vals)})")
print(f"Difference: {g_vals.mean() - t_vals.mean():.3f}")

**What you should see [REAL data]:** Generic messages produce equal or higher agreement than targeted ones on most issues. The participant-level t-test confirms the difference is significant (generic > targeted). Even with GPT-4 generating unique messages for each demographic profile, personalization does not beat a single strong generic argument.

**So far:** All three papers tested persuasion targeting (changing minds). The answer is clear: it does not work in politics. But campaigns care about two things: persuasion AND turnout. What about mobilization?

---
## Section 4: Aggarwal et al. (2023) [SIMULATED]

**Paper:** "A 2 Million-Person, Campaign-Wide Field Experiment Shows How Digital Advertising Affects Voter Turnout." *Nature Human Behaviour* 7: 332-341.

**Design:** $8.9M digital ad campaign randomized among 2 million "persuadable" voters in 5 battleground states during the 2020 US presidential election. Treatment group received 8 months of anti-Trump / pro-Biden digital ads (average 754 impressions per person). Outcome: **turnout** from the voter file.

**Data:** Individual-level voter file data is not publicly available (privacy restrictions on voter records). We **simulate** data calibrated to the paper's reported estimates: ATE = -0.06pp (SE 0.12), difference-in-CATEs by Trump Support Score = 0.7pp (p = 0.036).

**The question:** Persuasion targeting fails. Does **mobilization** targeting work?

In [ ]:
# ============================================================
# [SIMULATED] Aggarwal et al. (2023): Campaign-level field experiment
# ============================================================
# We simulate 2M voters with Trump Support Scores (TSS) and turnout,
# calibrated to the paper's reported estimates.
#
# Key reported numbers:
#   ATE on turnout: -0.06pp (SE 0.12, p = 0.60)
#   CATE for TSS 30-40 (Biden leaners): +0.4pp (SE 0.2)
#   CATE for TSS 60-70 (Trump leaners): -0.3pp (SE 0.3)
#   Difference-in-CATEs: 0.7pp (p = 0.036)

np.random.seed(2020)

N = 2_000_000
treat_frac = 0.50  # roughly half treated

# Trump Support Score: roughly uniform 20-80 (the "persuadable" range)
tss = np.random.uniform(20, 80, N)

# Treatment assignment (individual-level randomization)
treated = np.random.rand(N) < treat_frac

# Baseline turnout varies by TSS (from Figure 3: ~50-58% overall)
base_turnout_prob = 0.48 + 0.002 * (tss - 50)  # slight positive slope

# Treatment effect varies linearly with TSS
# Calibrated so: TSS=35 -> +0.004, TSS=65 -> -0.003, TSS=50 -> ~+0.0005
# slope: (0.004 - (-0.003)) / (35 - 65) = -0.000233 per TSS point
# intercept at TSS=50: ~+0.0005
cate = 0.0005 - 0.000233 * (tss - 50)

# Generate turnout
turnout_prob = base_turnout_prob + treated * cate
turnout = (np.random.rand(N) < turnout_prob).astype(int)

df_agg = pd.DataFrame({
    'tss': tss,
    'treated': treated,
    'turnout': turnout
})

print(f"[SIMULATED] {N:,} voters, {treated.sum():,} treated, {(~treated).sum():,} control")
print(f"[SIMULATED] Overall turnout: {turnout.mean():.4f}")
print(f"[SIMULATED] Turnout (treated): {turnout[treated].mean():.4f}")
print(f"[SIMULATED] Turnout (control): {turnout[~treated].mean():.4f}")
ate = turnout[treated].mean() - turnout[~treated].mean()
print(f"[SIMULATED] ATE: {ate:.4f} ({ate*100:.2f} pp)")

In [ ]:
# [SIMULATED] Replication of Figure 1: ATE and CATEs by subgroup
# The paper's Figure 1 shows ATEs by age, gender, race, margin, party, TSS

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Overall ATE (replicates the "ATE" row of Figure 1)
ax = axes[0]
n_t = df_agg[df_agg['treated']]['turnout']
n_c = df_agg[~df_agg['treated']]['turnout']
ate = n_t.mean() - n_c.mean()
se = np.sqrt(n_t.var()/len(n_t) + n_c.var()/len(n_c))

ax.errorbar([ate*100], [0], xerr=[[1.96*se*100], [1.96*se*100]],
            fmt='D', color='black', markersize=12, capsize=8, linewidth=2)
ax.axvline(0, color='black', linestyle='--', alpha=0.3)
ax.set_yticks([0])
ax.set_yticklabels(['ATE\n(all voters)'], fontsize=12)
ax.set_xlabel('Treatment Effect on Turnout (percentage points)')
ax.set_title(f'[SIMULATED] Overall ATE = {ate*100:.2f}pp (SE = {se*100:.2f})')
ax.set_xlim(-0.5, 0.5)

# Panel 2: CATEs by TSS bins (replicates the "Trump support" rows of Figure 1)
ax2 = axes[1]
tss_bins = [(30, 40, 'TSS 30-40\n(Biden leaners)'),
            (40, 50, 'TSS 40-50'),
            (50, 60, 'TSS 50-60'),
            (60, 70, 'TSS 60-70\n(Trump leaners)')]
colors_tss = [BLUE, GRAY, GRAY, RED]

for i, (lo, hi, label) in enumerate(tss_bins):
    mask = (df_agg['tss'] >= lo) & (df_agg['tss'] < hi)
    t = df_agg[mask & df_agg['treated']]['turnout']
    c = df_agg[mask & ~df_agg['treated']]['turnout']
    cate_est = t.mean() - c.mean()
    se_cate = np.sqrt(t.var()/len(t) + c.var()/len(c))

    ax2.errorbar([cate_est*100], [i], xerr=[[1.96*se_cate*100], [1.96*se_cate*100]],
                 fmt='D', color=colors_tss[i], markersize=10, capsize=6, linewidth=2)
    ax2.text(cate_est*100 + 1.96*se_cate*100 + 0.05, i,
             f'{cate_est*100:.2f}pp (SE={se_cate*100:.2f})', fontsize=9, va='center')

ax2.axvline(0, color='black', linestyle='--', alpha=0.3)
ax2.set_yticks(range(len(tss_bins)))
ax2.set_yticklabels([b[2] for b in tss_bins], fontsize=10)
ax2.set_xlabel('CATE on Turnout (percentage points)')
ax2.set_title('[SIMULATED] CATEs by Trump Support Score')

plt.suptitle('Aggarwal et al. (2023): $8.9M campaign, 2M voters\n'
             'Overall null, but differential mobilization by partisanship',
             fontsize=13, y=1.04)
plt.tight_layout()
plt.show()

# Compute difference-in-CATEs
biden = df_agg[(df_agg['tss'] >= 30) & (df_agg['tss'] < 40)]
trump = df_agg[(df_agg['tss'] >= 60) & (df_agg['tss'] < 70)]
cate_biden = biden[biden['treated']]['turnout'].mean() - biden[~biden['treated']]['turnout'].mean()
cate_trump = trump[trump['treated']]['turnout'].mean() - trump[~trump['treated']]['turnout'].mean()
dic = cate_biden - cate_trump
print(f"\nDifference-in-CATEs (TSS 30-40 vs TSS 60-70): {dic*100:.2f}pp")
print(f"Biden leaners CATE: +{cate_biden*100:.2f}pp")
print(f"Trump leaners CATE: {cate_trump*100:.2f}pp")
print(f"The campaign mobilized Biden supporters and demobilized Trump supporters.")

In [ ]:
# [SIMULATED] Replication of Figure 3: Turnout by TSS bins, treatment vs control
# The paper's Figure 3 shows turnout rates in 1-point TSS bins

fig, ax = plt.subplots(figsize=(12, 6))

bin_edges = np.arange(25, 76, 2)  # 2-point bins for smoother curves
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

treat_rates, ctrl_rates = [], []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (df_agg['tss'] >= lo) & (df_agg['tss'] < hi)
    t = df_agg[mask & df_agg['treated']]['turnout'].mean()
    c = df_agg[mask & ~df_agg['treated']]['turnout'].mean()
    treat_rates.append(t)
    ctrl_rates.append(c)

ax.plot(bin_centers, np.array(ctrl_rates)*100, 'o-', color=GRAY, markersize=4,
        label='Control', linewidth=1.5, alpha=0.8)
ax.plot(bin_centers, np.array(treat_rates)*100, 's-', color=BLUE, markersize=4,
        label='Treatment (ads)', linewidth=1.5, alpha=0.8)

# Shade the key TSS regions
ax.axvspan(30, 40, alpha=0.08, color=BLUE, label='Biden leaners (TSS 30-40)')
ax.axvspan(60, 70, alpha=0.08, color=RED, label='Trump leaners (TSS 60-70)')

ax.set_xlabel('Trump Support Score', fontsize=12)
ax.set_ylabel('Turnout Rate (%)', fontsize=12)
ax.set_title('[SIMULATED] Aggarwal et al. Figure 3: Turnout by Trump Support Score\n'
             'Treatment lifts Biden-leaner turnout, suppresses Trump-leaner turnout',
             fontsize=13)
ax.legend(loc='upper left', fontsize=10)

# Annotate the crossing
ax.annotate('Ads increase\nBiden-leaner turnout', xy=(35, 49.5), fontsize=10,
            color=BLUE, ha='center', fontweight='bold')
ax.annotate('Ads decrease\nTrump-leaner turnout', xy=(65, 52.5), fontsize=10,
            color=RED, ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("This replicates the shape of the paper's Figure 3:")
print("  - Treatment and control lines nearly overlap (tiny effects)")
print("  - But the gap reverses sign: positive for Biden leaners, negative for Trump leaners")
print("  - This is differential mobilization, not persuasion")

---
## Section 4: Aggarwal et al. (2023) -- Mobilization targeting [REAL]

**Data:** 1,993,917 voters from a $8.9M campaign-wide field experiment during the 2020 presidential election (Harvard Dataverse `doi:10.7910/DVN/YMKVA1`). Treatment = 8 months of digital ads (avg 754 impressions per person). Outcome = voted in 2020 (from voter file). Pre-registered CATE analysis by Trump Support Score (TSS).

**The question:** Targeting fails for persuasion. Does it work for mobilization (turnout)?

In [ ]:
# [REAL] Load Aggarwal et al. (2023) aggregated data
agg = pd.read_csv(DATA + 'aggarwal-by-decile.csv')
buckets = pd.read_csv(DATA + 'aggarwal-by-bucket.csv')

print(f"[REAL] Loaded turnout data aggregated by Trump Support Score")
print(f"  {agg['n'].sum():,} total voters across treatment and control")
print(f"  TSS range: {agg['tss_decile'].min()} (strong Biden) to {agg['tss_decile'].max()} (strong Trump)")
print(f"\nTurnout by bucket:")
print(buckets.to_string(index=False))

In [ ]:
# Replication of Figure 2: CATEs by Trump Support Score
# This is the paper's headline finding: differential mobilization

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Turnout by TSS (treated vs control)
ax = axes[0]
ctrl = agg[agg['treat_num'] == 0].sort_values('tss_decile')
treat = agg[agg['treat_num'] == 1].sort_values('tss_decile')

ax.plot(ctrl['tss_decile'], ctrl['turnout'], 'o-', color=GRAY, markersize=4,
        label=f'Control (n={ctrl["n"].sum():,})', alpha=0.8)
ax.plot(treat['tss_decile'], treat['turnout'], 's-', color=BLUE, markersize=4,
        label=f'Treated (n={treat["n"].sum():,})', alpha=0.8)
ax.set_xlabel('Trump Support Score (30=Biden, 70=Trump)')
ax.set_ylabel('Turnout Rate')
ax.set_title('[REAL] Aggarwal Fig. 2a: Turnout by Trump Support')
ax.legend(fontsize=10)

# Panel B: Treatment effect (CATE) by TSS
ax2 = axes[1]
merged = ctrl[['tss_decile', 'turnout', 'n', 'se']].rename(
    columns={'turnout': 'ctrl_turnout', 'n': 'ctrl_n', 'se': 'ctrl_se'})
merged = merged.merge(
    treat[['tss_decile', 'turnout', 'n', 'se']].rename(
        columns={'turnout': 'treat_turnout', 'n': 'treat_n', 'se': 'treat_se'}),
    on='tss_decile')
merged['cate'] = merged['treat_turnout'] - merged['ctrl_turnout']
merged['cate_se'] = np.sqrt(merged['ctrl_se']**2 + merged['treat_se']**2)

ax2.fill_between(merged['tss_decile'],
                 merged['cate'] - 1.96*merged['cate_se'],
                 merged['cate'] + 1.96*merged['cate_se'],
                 alpha=0.2, color=BLUE)
ax2.plot(merged['tss_decile'], merged['cate'], 'o-', color=BLUE, markersize=4)
ax2.axhline(0, color='black', linestyle='--', alpha=0.3)
ax2.axvline(50, color=GRAY, linestyle=':', alpha=0.5)
ax2.set_xlabel('Trump Support Score')
ax2.set_ylabel('CATE (Treated - Control Turnout)')
ax2.set_title('[REAL] Aggarwal Fig. 2b: Differential Mobilization')

# Annotate
ax2.text(35, ax2.get_ylim()[1]*0.8, 'Biden leaners:\nhigher turnout', color=GREEN, fontsize=10, ha='center')
ax2.text(65, ax2.get_ylim()[0]*0.8, 'Trump leaners:\nlower turnout', color=RED, fontsize=10, ha='center')

plt.suptitle('Aggarwal et al. (2023): $8.9M Field Experiment, 2M Voters', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Compute the difference-in-CATEs
biden_cate = buckets[(buckets['treat_num']==1) & (buckets['tss_bucket'].str.contains('Biden lean'))]['turnout'].values[0] - \
             buckets[(buckets['treat_num']==0) & (buckets['tss_bucket'].str.contains('Biden lean'))]['turnout'].values[0]
trump_cate = buckets[(buckets['treat_num']==1) & (buckets['tss_bucket'].str.contains('Trump lean'))]['turnout'].values[0] - \
             buckets[(buckets['treat_num']==0) & (buckets['tss_bucket'].str.contains('Trump lean'))]['turnout'].values[0]

overall_ate = treat['turnout'].mean() - ctrl['turnout'].mean()

print(f"\nOverall ATE on turnout: {overall_ate:.4f} (essentially zero)")
print(f"CATE for Biden leaners (TSS 30-40): {biden_cate:+.4f}")
print(f"CATE for Trump leaners (TSS 60-70): {trump_cate:+.4f}")
print(f"Difference-in-CATEs: {biden_cate - trump_cate:.4f}")
print(f"\n$8.9M bought differential mobilization of ~0.7pp (p=0.036 in paper).")

**What you should see [REAL data]:** Panel A shows treated and control turnout curves nearly overlapping (null ATE). Panel B shows the CATE by Trump Support Score: positive for Biden leaners (ads increased their turnout) and negative for Trump leaners (ads decreased their turnout). The difference-in-CATEs is ~0.7pp.

This is the mobilization exception: you can't change minds, but you can change who shows up. The effect is small even at $8.9M, which puts the Russian Facebook ad narrative ($150K spend) in perspective.

**What you should see [SIMULATED data, calibrated to reported estimates]:**

1. **Overall ATE:** Near zero. The $8.9M campaign did not change overall turnout.
2. **CATEs by TSS:** Biden leaners (TSS 30-40) show a small positive effect (~+0.4pp); Trump leaners (TSS 60-70) show a small negative effect (~-0.3pp). The difference-in-CATEs is ~0.7pp.
3. **Turnout curves:** Treatment and control lines nearly overlap, but the gap reverses sign at the midpoint. This is differential mobilization: the ads got Biden supporters to show up and Trump supporters to stay home.

**Why this matters:** All prior papers measured persuasion (changing minds). Aggarwal et al. show that even when you cannot change minds, you can change who shows up. But the effects are small even at campaign scale ($8.9M, 754 impressions per person). The popular narrative that Russia's $150K Facebook ad spend in 2016 could have swung the election is implausible.

---

## Key Takeaways

1. **Coppock (2020):** Meta-analysis of 354 CATEs: no subgroup, no ad type, no context predicts larger effects. Small effects are uniformly small.
2. **Tappin (2023):** Party ID captures most predictable variation in persuadability. Additional covariates add near-zero marginal value.
3. **Simchon (2024):** Even GPT-4 cannot write its way past partisan identity. Generic messages beat targeted ones.
4. **Aggarwal (2023):** Persuasion targeting fails, but mobilization targeting works at the margin. Differential turnout of 0.7pp from an $8.9M campaign.

**The pattern:** Targeting works for low-stakes product marketing (Matz 2017) but fails for political persuasion because partisan identity dominates. The one exception: mobilization (getting your people to show up), where small differential effects are detectable at massive scale.

**Next class:** If static targeting cannot save you, can *adaptive* algorithms do better? Multi-Armed Bandits learn while deploying.